# Concatenación y homogeneización de bases de isótopos foliares

Este notebook combina las 4 bases de datos (`Galeras_cellulose.xlsx`, `SV_cellulose.xlsx`, `Sumaco_cellulose.xlsx`, `iWUE_OYC_GUA.xlsx`) en una sola tabla homogeneizada, añade la **elevación por parcela** a partir de `elevation_dict.py`, y genera un **reporte de errores/inconsistencias** que deben revisarse manualmente.

**Qué hace el script, en orden:**
1. Carga cada archivo.
2. Estandariza el nombre de columnas clave (`Sample_ID`, `Plot`, `Site`, etc.).
3. Limpia el código de parcela (`Plot`): quita el prefijo `p`, ceros a la izquierda y el `.0` de los que venían como decimal.
4. Homogeneiza las columnas de isótopos que venían en dos unidades distintas (fracción 0–1 vs. porcentaje): `[C]%`, `[C_b]%`, `[N]`.
5. Concatena las 4 tablas en una sola.
6. Añade la columna `Elevation_m` cruzando `(Site, Plot)` contra `elevation_dict`.
7. Corre 3 controles de calidad (QC) y marca en `QC_flag` las filas que necesitan revisión manual.
8. Guarda el resultado en un Excel con 2 hojas: `Datos_concatenados` (todo) y `Errores_a_revisar` (solo las filas marcadas).

## 1. Librerías

In [164]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

## 2. Diccionario de elevación por parcela

Tomado de `elevation_dict.py`. Las llaves son tuplas `(Sitio, Parcela)`. **Importante:** este diccionario solo cubre los sitios `Galeras`, `Oyacachi` y `Sumaco`. Los sitios `SV` y `Guacamayos` (presentes en tus archivos `SV_cellulose.xlsx` e `iWUE_OYC_GUA.xlsx`) **no tienen elevación asignada todavía** — quedarán marcados en el reporte de errores para que agregues esos valores manualmente.

In [165]:
elevation_dict = {
    # Galeras
    ("Galeras", 20): 1090, ("Galeras", 21): 1080, ("Galeras", 22): 1060,
    ("Galeras", 23): 1090, ("Galeras", 24): 1110, ("Galeras", 29): 1050,
    ("Galeras", 47): 1000, ("Galeras", 48): 1130, ("Galeras", 49): 1080,
    ("Galeras", 28): 1597,
    ("Galeras", 25): 1450, ("Galeras", 26): 1560, ("Galeras", 50): 1557,
    ("Galeras", 60): 1570, ("Galeras", 61): 1560, ("Galeras", 62): 1590,
    # Oyacachi
    ("Oyacachi", 81): 4000, ("Oyacachi", 82): 4000, ("Oyacachi", 83): 4000,
    ("Oyacachi", 84): 3500, ("Oyacachi", 85): 3500, ("Oyacachi", 86): 3500,
    # Sumaco
    ("Sumaco", 9): 2000, ("Sumaco", 10): 2000, ("Sumaco", 11): 2000,
    ("Sumaco", 12): 2000, ("Sumaco", 13): 2000,
    ("Sumaco", 16): 1500, ("Sumaco", 18): 1500,
    #Guacamayos 
      ("Guacamayos", 6): 2000, ("Guacamayos", 7): 1940, ("Guacamayos", 8): 1995,
        ("Guacamayos", 55): 2000,("Guacamayos", 27): 1990,
    # Selva Viva
    ("Selva Viva", 68): 502, ("Selva Viva", 69): 508, ("Selva Viva", 70): 508,
    ("Selva Viva", 72): 453, ("Selva Viva", 74):470
}

## 3. Funciones auxiliares

- `clean_plot`: normaliza el código de parcela a entero (quita `p`, ceros a la izquierda, `.0`).
- `homogenize_pct`: detecta si un valor está en fracción (0–1) o en porcentaje, y homogeneiza todo a **porcentaje**. Regla: si `abs(valor) < 1` se asume fracción y se multiplica por 100; si ya es `>= 1` se asume que ya está en %. Esta regla se validó revisando duplicados del mismo `Sample_ID` con y sin homogeneizar (p.ej. `p22-550` con 0.474 y 47.2 para la misma muestra: son la misma medición en dos unidades distintas).

In [166]:
def clean_plot(v):
    """Normaliza el codigo de parcela a entero."""
    if pd.isna(v):
        return pd.NA
    s = str(v).strip().lower()
    if s.endswith('.0'):
        s = s[:-2]
    if s.startswith('p'):
        s = s[1:]
    s = s.lstrip('0')
    if s == '':
        s = '0'
    try:
        return int(s)
    except ValueError:
        return pd.NA

def homogenize_pct(series):
    """Pasa fracciones (abs<1) a porcentaje (x100). Deja igual lo que ya esta en %.
    Los negativos con abs>=1 no se tocan: son fisicamente imposibles y se marcan luego como error."""
    def fix(v):
        if pd.isna(v):
            return v
        if abs(v) < 1:
            return v * 100
        return v
    return series.apply(fix)

def implied_plot(sample_id):
    """Extrae la parcela implicita en el Sample_ID (ej. 'p09-230' -> 9) para cruzarla con la columna Plot."""
    if pd.isna(sample_id):
        return pd.NA
    m = re.match(r'^p0*([0-9]+)-', str(sample_id).strip().lower())
    return int(m.group(1)) if m else pd.NA

TYPE_MAP = {'herbarium': 'leaf herbarium'}  # unifica nomenclatura de 'Type'

## 4. Cargar y estandarizar cada base

En esta sección se define, para cada archivo:
- `Site` (sitio de muestreo), tomado del nombre del archivo, o de la columna `Site` si ya existía (caso `iWUE_OYC_GUA.xlsx`, que trae `Oyacachi` y `Guacamayos`).
- Limpieza de `Plot` y de las columnas de isótopos/porcentaje.
- Columnas redundantes o duplicadas que se eliminan (con la razón indicada en el comentario).

### 4.1 Galeras

In [167]:
g = pd.read_excel(r"Galeras/Galeras_cellulose.xlsx")

g['Site'] = 'Galeras'
g['Plot'] = g['Plot'].apply(clean_plot)
for c in ['[C]%', '[C_b]%', '[N]']:
    g[c] = homogenize_pct(g[c])
g['Type'] = g['Type'].replace(TYPE_MAP)
g['source_file'] = 'Galeras_cellulose.xlsx'
g = g.rename(columns={'Sample-ID': 'Sample_ID'})
g.head(2)

,Unnamed: 0,Sample_ID,[C]%,δ13C (‰ v.s.V-PDB),Type,[N],[C_b]%,δ15N (‰ v.s. V-PDB),bulk_δ13C (‰ v.s.V-PDB),Year,old_treeID,description,Type 2,Plot,new_TreeID,family,JH herbarium collections,genus,species,Site,source_file
0,0,p23-557,45.552995,-29.289590,leaf pulverized,1.912373,48.553106,1.960678,-31.458731,2006,557.0,NaN,NaN,23,NaN,Quiinaceae,3737,Lacunaria,crenata,Galeras,Galeras_cellulose.xlsx
1,1,p23-557,43.360479,-29.427788,leaf pulverized,1.912373,48.553106,1.960678,-31.458731,2006,557.0,NaN,NaN,23,NaN,Quiinaceae,3737,Lacunaria,crenata,Galeras,Galeras_cellulose.xlsx


### 4.2 SV

In [168]:
sv = pd.read_excel(r"Selva Viva\SV_cellulose.xlsx")

sv['Site'] = 'SV'
sv['Plot'] = sv['Plot'].apply(clean_plot)
for c in ['[C]%', '[C_b]%', '[N]']:
    sv[c] = homogenize_pct(sv[c])
sv['Type'] = sv['Type'].replace(TYPE_MAP)
sv['source_file'] = 'SV_cellulose.xlsx'
sv = sv.rename(columns={'Sample-ID': 'Sample_ID'})
sv["Site"] = "Selva Viva"
if 'sample-ID' in sv.columns:            # columna duplicada (minuscula), redundante con Sample_ID
    sv = sv.drop(columns=['sample-ID'])
sv.head(2)

,Unnamed: 0,Sample_ID,[C]%,δ13C (‰ v.s.V-PDB),Year,Plot,Type,numeroCampo,description,old_treeID,[N],[C_b]%,δ15N (‰ v.s. V-PDB),bulk_δ13C (‰ v.s.V-PDB),new_TreeID,family,JH herbarium collections,genus,species,Site,source_file
0,110,jh-p70-3904,39.990000,-31.980000,2006,70,leaf herbarium,3904.0,plot 70 species 2,NaN,NaN,NaN,NaN,NaN,3604.0,Meliaceae,NaN,Guarea,NaN,Selva Viva,SV_cellulose.xlsx
1,14,jh-3939,43.988841,-32.233186,2006,68,leaf herbarium,3939.0,plot 68 species 15,NaN,NaN,NaN,NaN,NaN,3604.0,Meliaceae,NaN,Guarea,NaN,Selva Viva,SV_cellulose.xlsx


### 4.3 Sumaco

In [169]:
su = pd.read_excel((r"Sumaco\Sumaco_cellulose.xlsx"))
su['Site'] = 'Sumaco'
su['Plot'] = su['Plot'].apply(clean_plot)
su['[C]%'] = homogenize_pct(su['[C]%'])   # este archivo no trae [C_b]% ni [N]
su['source_file'] = 'Sumaco_cellulose.xlsx'
su = su.rename(columns={'Sample-ID': 'Sample_ID'})
su.head(2)

,Unnamed: 0,Sample_ID,family,genus,species,[C]%,δ13C (‰ v.s.V-PDB),Plot,Year,old_treeID,new_TreeID,JH herbarium collections,Type,Site,source_file
0,0,p09-225,Moraceae,Ficus,quijosana,42.578188,-28.691043,9,2006,225,NaN,NaN,leaf pulverized,Sumaco,Sumaco_cellulose.xlsx
1,1,p09-230,Lauraceae,Ocotea,insularis,45.456318,-29.289822,1,2006,230,NaN,3343.0,leaf pulverized,Sumaco,Sumaco_cellulose.xlsx


### 4.4 iWUE (Oyacachi / Guacamayos)

In [170]:
iw = pd.read_excel(r"Oyacachi-Guacamayos\iWUE_OYC_GUA.xlsx")
iw['Plot'] = iw['Plot'].apply(clean_plot)
for c in ['[C]%', '[C_b]%', '[N]']:
    iw[c] = homogenize_pct(iw[c])
iw['Type'] = iw['Type'].replace(TYPE_MAP)
iw['source_file'] = 'iWUE_OYC_GUA.xlsx'
iw = iw.rename(columns={'Sample-ID': 'Sample_ID',
                         'species2': 'species',
                         'herbarium specimen': 'JH herbarium collections'})
if '[C]%_new' in iw.columns:   # es un duplicado exacto de [C]% donde existe -> se elimina
    iw = iw.drop(columns=['[C]%_new'])
for c in ['species3', 'species4', 'species5']:  # desgloses finos (cf/aff), poco poblados -> se descartan
    if c in iw.columns:
        iw = iw.drop(columns=[c])
iw.head(2)

,Unnamed: 0,new_TreeID,old_treeID,Plot,Site,family,leaf_sample_ID_2011,JH herbarium collections,[C]%,δ13C (‰ v.s.V-PDB),[N],δ15N (‰ v.s. V-PDB),[C_b]%,bulk_δ13C (‰ v.s.V-PDB),Year,Type,Sample_ID,genus_species,genus,species,comment,source_file
0,0,4039.0,4039.0,82,Oyacachi,Solanaceae,56077.0,4194.0,45.059988,-24.65,1.518827,-2.154079,44.042720,-27.419388,2011,leaf pulverized,p82-4039,Solanum stenophyllum,Solanum,stenophyllum,NaN,iWUE_OYC_GUA.xlsx
1,1,8016.0,4020.0,81,Oyacachi,Rosaceae,56069.0,NaN,45.781855,-26.37,2.984893,0.134936,45.888802,-28.960106,2011,leaf pulverized,p81-4020,Polylepis pauta,Polylepis,pauta,NaN,iWUE_OYC_GUA.xlsx


## 5. Concatenar las 4 bases

Se concatenan por nombre de columna (`sort=False` conserva el orden), rellenando con `NaN` las columnas que no existen en todas las bases (por ejemplo `PlotID` solo existe en Sumaco).

In [171]:
all_df = pd.concat([g, sv, su, iw], ignore_index=True, sort=False)
all_df = all_df.drop(columns=['Unnamed: 0'], errors='ignore')
print('Filas totales:', len(all_df))
all_df['Site'].value_counts(dropna=False)


# Diccionario con las correcciones
plot_fix = {
    "p28-8878": 28,
    "p60-8885": 60,
    "p48-3698": 48,
    "p68-8878": 68,
    "p68-8885": 68,
    "p10-252": 10,
    "p12-304": 12,
    "p13-315": 13,
    "p09-230": 9, #decia 1 en lugar de 9,
    "p62-8669":48, #no coincide el plot pero verificado manualmente
    "p28-8974":23, # 

}

# Suponiendo que la columna con esos códigos se llama "Sample"
for sample, plot in plot_fix.items():
    all_df.loc[all_df["Sample_ID"] == sample, "Plot"] = plot

# (Opcional) dejar Plot como entero
##df["Plot"] = df["Plot"].astype("Int64")
# Muestras afectadas

# Columnas a intercambiar
col1 = "[C_b]%"
col2 = "δ15N (‰ v.s. V-PDB)"
# p82-4044 únicamente para 2011
mask1 = (all_df["Sample_ID"] == "p82-4044") & (all_df["Year"] == 2011)
mask2 = all_df["Sample_ID"] == "p81-8023"

# Intercambiar los valores
for mask in [mask1, mask2]:
    all_df.loc[mask, [col1, col2]] = all_df.loc[mask, [col2, col1]].to_numpy()


Filas totales: 669


## 6. Añadir elevación por parcela

Se cruza `(Site, Plot)` contra `elevation_dict`. Donde no hay coincidencia, `Elevation_m` queda en `NaN`.

In [172]:
def get_elev(row):
    return elevation_dict.get((row['Site'], row['Plot']), np.nan)

all_df['Elevation_m'] = all_df.apply(get_elev, axis=1)
all_df['Elevation_m'].isna().sum()

0

## 7. Control de calidad (QC)

Se marca cada fila en la columna `QC_flag` con uno o más de estos motivos:

1. **`[C_b]% negativo`**: hay 2 filas (site Oyacachi) donde el carbono a granel homogeneizado da negativo — dato imposible, seguramente error de captura/instrumento. Revisar el dato crudo.
3. **`Plot vacio/no interpretable`**: la celda de parcela venía vacía o con un valor no interpretable como número.
4. **`Plot no coincide con prefijo del Sample_ID`**: el código de parcela en `Sample_ID` (ej. `p09-230` implica parcela 9) no coincide con el valor de la columna `Plot` de esa fila (ej. decía `01`). Son 3 filas — probablemente errores de tipeo en la planilla original.

In [173]:
print(all_df[all_df["Type"].isna()]["Sample_ID"])#que filas les falta inofmracion, segun esto estas pueden ir com 
mask = all_df["Type"].isna() #creo una mascara con los que son nan
all_df.loc[mask, "Type"] = "leaf pulverized" #se añade leaf pulverized

38      p23-565
39      p23-578
40      p23-566
41      p29-792
392    p68-8878
398    p68-8885
658     p06-157
659     p27-669
660     p27-682
661     p27-687
662     p27-694
663     p27-695
664     p27-696
665     p27-704
666     p27-708
667     p27-712
668     p27-714
Name: Sample_ID, dtype: object


In [174]:
all_df['QC_flag'] = ''

# 1. [C_b]% fisicamente imposible
neg_mask = all_df['[C_b]%'] < 0
all_df.loc[neg_mask, 'QC_flag'] += '[C_b]% negativo (revisar manualmente); '

# 2. Sin elevacion asignada
elev_missing_mask = all_df['Elevation_m'].isna() & all_df['Plot'].notna()
all_df.loc[elev_missing_mask, 'QC_flag'] += 'Sin elevacion en elevation_dict; '

# 3. Plot vacio
plot_missing_mask = all_df['Plot'].isna()
all_df.loc[plot_missing_mask, 'QC_flag'] += 'Plot vacio/no interpretable; '

# 4. Plot no coincide con el Sample_ID
all_df['Plot_from_ID'] = all_df['Sample_ID'].apply(implied_plot)
mismatch_mask = (all_df['Plot_from_ID'].notna()) & (all_df['Plot'].notna()) & (all_df['Plot_from_ID'] != all_df['Plot'])
all_df.loc[mismatch_mask, 'QC_flag'] += 'Plot no coincide con prefijo del Sample_ID; '

print('Filas marcadas:', (all_df['QC_flag'] != '').sum(), 'de', len(all_df))

Filas marcadas: 2 de 669


## 8. Reordenar columnas y guardar resultados

Se guarda un Excel con dos hojas:
- `Datos_concatenados`: la tabla completa homogeneizada.
- `Errores_a_revisar`: solo las filas con `QC_flag` distinto de vacío, para que las revises manualmente.

In [ ]:
front_cols = ['Site', 'Plot', 'Elevation_m', 'Sample_ID', 'Year', 'Type',
              'family', 'genus', 'species',
              '[C]%', 'δ13C (‰ v.s.V-PDB)',
              '[C_b]%', 'bulk_δ13C (‰ v.s.V-PDB)',
              '[N]', 'δ15N (‰ v.s. V-PDB)',
              'QC_flag']
other_cols = [c for c in all_df.columns if c not in front_cols]
all_df = all_df[front_cols + other_cols]

out_path = 'isotopos_concatenado.xlsx'
report_rows = all_df[all_df['QC_flag'] != ''][
    ['Sample_ID','Site','Plot','Plot_from_ID','Elevation_m','[C]%','[C_b]%',
     'δ15N (‰ v.s. V-PDB)','QC_flag','source_file']
]

all_df = all_df.drop(columns={"genus_species","comment","Plot_from_ID","Type2"})
#en la columna comment solo hay un comentario de un reeemplzao de un sp de una Aniba coto, y Type2 era para ver la Miscelanea

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    all_df.to_excel(writer, sheet_name='Datos_concatenados', index=False)
    report_rows.to_excel(writer, sheet_name='Errores_a_revisar', index=False)

print(f'Guardado: {out_path}')
report_rows

Guardado: isotopos_concatenado.xlsx


,Sample_ID,Site,Plot,Plot_from_ID,Elevation_m,[C]%,[C_b]%,δ15N (‰ v.s. V-PDB),QC_flag,source_file
167,p62-8669,Galeras,48,62,1130,51.550183,NaN,NaN,Plot no coincide con prefijo del Sample_ID;,Galeras_cellulose.xlsx
205,p28-8974,Galeras,23,28,1090,40.540000,NaN,NaN,Plot no coincide con prefijo del Sample_ID;,Galeras_cellulose.xlsx


## 9. Resumen de lo que debes revisar manualmente

- **Corregido: Elevación faltante para `SV` y `Guacamayos`**: agrega esas parcelas a `elevation_dict` (313 filas de `SV` y 90 de `Guacamayos` quedaron sin elevación).
- **Corregido: 2 filas con `[C_b]%` negativo** (site Oyacachi, `p82-4044` y `p81-8023`), que además tienen δ15N fuera de rango (~42–43‰, cuando el resto de la base está entre -5 y +8‰). Revisar el dato crudo de laboratorio.
- **Corregido. 3 filas con parcela ambigua** (`Plot` no coincide con el prefijo del `Sample_ID`): `p62-8669`, `p28-8974`, `p09-230`.
- Un puñado de filas con `Plot` vacío que no se pudieron interpretar.